# 📦 1️⃣ Importaciones y carga de claves

In [ ]:
import os
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -----------------------------
# RUTAS CORRECTAS PARA JUPYTER
# -----------------------------

# Ruta del proyecto (sube 1 nivel desde /notebooks)
BASE_DIR = Path().resolve().parents[0]      # <-- esta es la clave

# Si tu notebook está en: 03_IA_Segunda_Guerra_Mundial/notebooks/
# Entonces BASE_DIR apunta a: 03_IA_Segunda_Guerra_Mundial/

PROCESSED_DIR = BASE_DIR / "data" / "processed"

CHUNKS_PATH = PROCESSED_DIR / "ww2_chunks.jsonl"
EMB_PATH = PROCESSED_DIR / "ww2_embeddings.npy"


NameError: name '__file__' is not defined

# 📄 2️⃣ Cargar chunks y embeddings

In [ ]:
def cargar_datos():
    chunks = []
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))

    emb_matrix = np.load(EMB_PATH)

    print(f"Chunks cargados: {len(chunks)}")
    print(f"Embeddings shape: {emb_matrix.shape}")

    return chunks, emb_matrix


chunks, emb_matrix = cargar_datos()


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\lucia\\Documents\\GitHub\\03_IA_Segunda_Guerra_Mundial\\notebooks\\data\\processed\\ww2_chunks.jsonl'

# 🧠 3️⃣ Función para crear embeddings de la pregunta

In [ ]:
def embed_text(text):
    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=text,
    )
    return np.array(resp.data[0].embedding, dtype="float32")


# 📏 4️⃣ Similitud coseno + recuperación de top-k fragmentos

In [ ]:
def cosine_sim(query_vec, docs_matrix):
    num = docs_matrix @ query_vec
    den = (np.linalg.norm(docs_matrix, axis=1) * np.linalg.norm(query_vec) + 1e-10)
    return num / den


def recuperar_chunks(query, k=5):
    q_emb = embed_text(query)
    sims = cosine_sim(q_emb, emb_matrix)

    idx = np.argsort(sims)[::-1][:k]

    resultados = []
    for i in idx:
        c = chunks[int(i)]
        resultados.append({
            "score": float(sims[i]),
            "text": c["text"],
            "source": c["source"],
            "page": c["page"]
        })

    return resultados


# 📝 5️⃣ Prompt conciso + llamada al modelo

In [ ]:
def generar_respuesta(query, contexto):
    # Construir fragmentos
    partes = []
    for i, c in enumerate(contexto):
        partes.append(
            f"[Fragmento {i+1} | Fuente: {c['source']} pág. {c['page']}]\n{c['text']}"
        )
    context_text = "\n\n".join(partes)

    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un historiador experto en la Segunda Guerra Mundial. "
                "Respondes siempre en español, de manera breve y directa, máximo 6 líneas. "
                "Tu única fuente es el contexto proporcionado. No inventes datos."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{context_text}\n\n"
                "Usa SOLO la información anterior. Si algo no aparece, responde: "
                "\"No existe información suficiente en los documentos para responder.\"\n\n"
                "Instrucciones:\n"
                "- Máximo 6 líneas.\n"
                "- Sin subtítulos.\n"
                "- Respuesta clara y concisa.\n"
                "- Añade citas al final: (Fuente: <PDF>, pág. X).\n\n"
                f"Pregunta: {query}"
            ),
        },
    ]

    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=mensajes,
        max_completion_tokens=300,
    )

    return resp.choices[0].message.content


# ❓ 6️⃣ Función final para preguntarle:

In [ ]:
def responder(query, k=5):
    contexto = recuperar_chunks(query, k)
    respuesta = generar_respuesta(query, contexto)
    return respuesta


# PREGUNTAR

In [ ]:
responder("¿Por qué fue importante la Operación Barbarroja?")
